# Decode the Bioreactor: from mechanistic to hybrid models

**Goal.** For each of the 12 batches to predict, only the day-0 sample, the feeding plan and the volume are known. We predict four quantities on days 1–13:

| Symbol | Column | Meaning | Unit |
|---|---|---|---|
| $\mathrm{VCD}$ | `vcd` | viable cell density | $10^6$ cells/mL |
| $\mathrm{TCD}$ | `tcd` | total cell density (viable + dead) | $10^6$ cells/mL |
| $S$ | `substrate` | substrate concentration | g/L |
| $P$ | `byproduct` | byproduct concentration | g/L |

**Strategy.** We start from the simplest possible prediction and add one idea at a time. Every section is one model = one upgrade. Each section ends with
1. its benchmark score (added to a running table),
2. an interactive viewer with all the models built so far,
3. a submission file ready to upload (`outputs/submission_M#_….csv`).

| Model | Name | What is new |
|---|---|---|
| M0 | Mean curve | no physics: average of the training batches, day by day |
| M1 | Logistic growth | first physics: growth slows down as the density rises |
| M2 | Cell model (daily rates) | + brake from cumulative load, + cell death; parameters from measured daily rates |
| M3 | Cell model (joint fit) | same equations, parameters fitted on the full curves |
| M4 | Cells + metabolite balances | + mass balances of substrate and byproduct with feed boluses (documented failure) |
| M5 | Hybrid: mechanistic + ML | M3 + machine-learning correction of its errors |
| M5b | Data-driven: mean + ML | M0 + the same correction (control experiment) |
| M6 | Grey-box capacity + ML | the capacity $\mathrm{VCD}_{\max}$ becomes batch-specific, predicted from batch features |
| M7 | Grey-box + feed shape | + features describing the shape of the feeding profiles |

The code lives in the library `src/`:

| File | Content |
|---|---|
| `data_handling.py` | loading, per-batch packages, features |
| `mechanistic.py` | differential equations, simulation, parameter fits |
| `models.py` | the models M0–M7 with a common `fit` / `predict` interface |
| `evaluate.py` | team benchmark, cross-validation, submission writer |
| `plotting.py` | plot style, interactive viewer, summary charts |

## 0. Setup
Set `REPO` to the folder that contains `src/`. Submissions are written to `REPO/outputs/`.

In [ ]:
import sys, warnings # path handling, warnings
from pathlib import Path # paths
import numpy as np # numerics
import pandas as pd # tables
import matplotlib.pyplot as plt # static plots
from IPython.display import display # show tables

REPO = Path("/home/renku/work/team-workspace/Alessia") # folder containing src/
sys.path.insert(0, str(REPO)) # make src importable
OUT = REPO / "outputs" # submission folder
OUT.mkdir(exist_ok=True) # create if missing
warnings.filterwarnings("ignore") # quiet integrator warnings

import src.data_handling as dh # data
import src.mechanistic as mech # equations
import src.models as mdl # models
import src.evaluate as ev # benchmark
import src.plotting as pl # plots

pl.setup() # plot style
pd.set_option("display.precision", 3) # table precision

## 1. Data
The file contains 61 batches: 49 with daily measurements (*labeled*) and 12 to predict, which have only the day-0 sample.

Two kinds of rows:
- **samples**: one per day, with VCD, TCD, substrate, byproduct and volume;
- **feeds**: additions of `feed_substrate`, `feed_b`, `feed_c`, `feed_d` at given times.

`Data()` loads everything once. It builds one *package* per batch (day-0 state, prediction times, measured curves, volume, feeding plan) and the feature tables used by the ML models.

In [ ]:
data = dh.Data() # load and prepare everything
print(f"labeled batches: {len(data.labeled)}, batches to predict: {len(data.to_predict)}") # counts
display(data.meas.head()) # first samples

All labeled batches over time. The red curves are the 10 batches the team keeps aside for the benchmark (section 2). The spread between batches, much more than the shape of each curve, is what makes the task hard.

In [ ]:
pl.overview(data) # all batches, test batches highlighted

## 2. Benchmark: how the models are compared

All team members use the same hold-out split:
- **test**: Batch2, 3, 11, 18, 19, 20, 23, 28, 39, 42 (10 batches);
- **train**: the other 39 labeled batches.

A model is fitted on the train batches only. Then it predicts the test batches from their day-0 sample and feeding plan.

**Step 1: normalized error.** For test batch $b$, target $c$ and sample $k$:
$$e_{b,c,k} = \frac{\hat y_{b,c}(t_k) - y_{b,c}(t_k)}{\sigma_c}$$
Here $\hat y$ is the prediction and $y$ the measurement. $\sigma_c$ is the standard deviation of target $c$ over all training samples after day 0. Dividing by $\sigma_c$ makes the four targets comparable, because they have different units and sizes. It is only a scale: it does not change which model is better on a given target.

**Step 2: mean squared error per batch and target** ($K_b$ = number of samples of batch $b$):
$$m_{b,c} = \frac{1}{K_b}\sum_{k=1}^{K_b} e_{b,c,k}^{\,2}$$

**Step 3: overall score.** This is the root of the mean over the $10 \times 4 = 40$ values:
$$S = \sqrt{\frac{1}{40}\sum_{b}\sum_{c} m_{b,c}}$$

**Per-target score**, shown in the tables: $S_c = \sqrt{\tfrac{1}{10}\sum_b m_{b,c}}$. The total relates to the per-target scores by $S^2 = \tfrac14\sum_c S_c^2$.

**How to read it.** Lower is better. $S = 0.30$ means the typical error is about 30% of the natural spread of that variable. Predicting a single constant number would give $S \approx 1$.

The scales $\sigma_c$:

In [ ]:
train_ids, test_ids = ev.split_benchmark(data) # team split
display(ev.target_sd(data, train_ids).to_frame("σ (train, after day 0)")) # normalization scales

Helper used in every model section: run the benchmark, store the result, write the submission (fitted on all 49 labeled batches) and show the running table.

In [ ]:
results, preds = {}, {} # benchmark results and predictions, by model

# benchmark one model, write its submission, show all results so far
def run(key, make_model, csv=None, show=None):
    r = ev.benchmark(make_model, data) # fit on 39, score on 10
    results[key] = r # store result
    preds[r["model"].name] = r["pred"] # store predictions for the viewer
    if csv: # submission only if a file name is given
        ev.write_submission(make_model, data, OUT / csv) # final fit on 49, predict the 12
    print(f"{r['model'].name}: score = {r['score']:.3f}") # summary line
    table = ev.results_table(results) # all results so far
    display(table if show is None else table[table.index.str.startswith(show)]) # all models, or only the selected ones
    return r

## M0: Mean curve
**Idea.** Ignore physics and batch information. For every day $d$, predict the average of the training batches on that day:
$$\hat y_{c}(d) = \frac{1}{B}\sum_{b \in \text{train}} y_{b,c}(d)$$
This is the reference: a model is useful only if it beats M0.

In [ ]:
r0 = run("M0", lambda: mdl.MeanCurve()) # benchmark + submission
pl.model_viewer(data, preds) # interactive viewer

## M1: Logistic growth
**Idea.** Cells divide, so their number grows in proportion to itself. Growth slows down as the culture becomes crowded.

Use totals, so that sampling and feeding (volume changes) do not create fake growth: $N = \mathrm{VCD}\cdot V$ (viable cells in the reactor).

**Why not plain exponential growth?** With exponential growth, $\ln N$ would be a straight line in time. The diagnostic below shows it bends early. The observed growth rate falls as the density rises:
$$\mu_{\text{obs}} = \frac{\ln N_{k+1} - \ln N_k}{t_{k+1} - t_k}$$

**Equations.**
$$\frac{dN}{dt} = \mu\, N, \qquad \mu = \mu_{\max}\left(1 - \frac{\mathrm{VCD}}{\mathrm{VCD}_{\max}}\right), \qquad \mathrm{VCD} = \frac{N}{V}$$
- $\mu_{\max}$ [1/day]: growth rate of an empty reactor;
- $\mathrm{VCD}_{\max}$ [$10^6$ cells/mL]: density at which growth stops.

**Parameters.** A straight line $\mu_{\text{obs}} = \mu_{\max} - (\mu_{\max}/\mathrm{VCD}_{\max})\,\mathrm{VCD}$ is fitted on the growth phase (day < 6.5): its intercept gives $\mu_{\max}$, its zero gives $\mathrm{VCD}_{\max}$. No death: $\mathrm{TCD}$ keeps the day-0 viability. Substrate and byproduct stay on the mean curve (M0).

In [ ]:
rates = mech.observed_rates(data.samples(train_ids)) # observed rates of the training batches
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True) # two diagnostics
for bid, g in rates.groupby("batch_id"): # each batch
    axes[0].plot(g["elapsed_time"], np.log(g["cells"]), color="#0B4F5C", alpha=0.35, lw=0.9) # ln N over time
axes[0].set(xlabel=pl.LABELS["elapsed_time"], ylabel="ln N  (N = VCD·V)", title="ln N is not a straight line") # labels
gr = rates[rates["elapsed_time"] < 6.5] # growth phase
axes[1].scatter(gr["vcd"], gr["mu_obs"], c=gr["elapsed_time"], cmap="viridis", s=12) # rate vs density
p1 = mech.logistic_from_rates(rates) # logistic parameters
x = np.linspace(0, gr["vcd"].max(), 50) # density grid
axes[1].plot(x, p1["mu_max"] * (1 - x / p1["vcd_max"]), color="#E4572E", lw=2, label="logistic fit") # fitted line
axes[1].set(xlabel=pl.LABELS["vcd"], ylabel=r"$\mu_{\mathrm{obs}}$ [1/day]", title="growth rate falls with density") # labels
axes[1].legend() # legend
plt.show() # draw
display(pd.Series(p1).to_frame("M1 parameters")) # fitted parameters

In [ ]:
r1 = run("M1", lambda: mdl.LogisticModel()) # benchmark + submission
pl.model_viewer(data, preds) # interactive viewer

**Result.** The logistic model is *worse* than M0 on VCD. The density brake alone cannot stop growth at the right time. Real cultures stop earlier and then decline, and this model has no mechanism for either.

## M2: Cell model from daily rates
**What is new.** Two mechanisms that M1 lacks.

1. **Cumulative brake.** Growth also slows with the load the culture has accumulated over time, measured by the integral of viable cell density:
$$\mathrm{IVCD}(t) = \int_0^t \mathrm{VCD}\,dt' \quad [10^6\ \text{cells·day/mL}]$$
2. **Death.** Viable cells die at a rate $k_d$ that rises with the accumulated load. Dead cells $D = (\mathrm{TCD}-\mathrm{VCD})\,V$ stay in the reactor.

**Equations.**
$$\frac{dN}{dt} = (\mu - k_d)\,N, \qquad \frac{dD}{dt} = k_d\,N, \qquad \frac{d\,\mathrm{IVCD}}{dt} = \mathrm{VCD}$$
$$\mu = \mu_{\max}\left(1 - \frac{\mathrm{VCD}}{\mathrm{VCD}_{\max}} - \frac{\mathrm{IVCD}}{\mathrm{IVCD}_{\max}}\right), \qquad k_d = k_{d,0} + k_{d,1}\,\mathrm{IVCD}$$

$N$ and $D$ are totals, so dilution by feeds does not change them. The VCD decline after the peak is mostly dilution: same cells, more volume.

**Parameters from measured rates.** For each interval between two samples:
$$k_{d,\text{obs}} = \frac{D_{k+1}-D_k}{\bar N\,(t_{k+1}-t_k)}, \qquad \mu_{g} = \mu_{\text{obs}} + k_{d,\text{obs}} \quad (\text{gross growth, births only})$$
Here $\bar N$ is the mean of $N$ over the interval. Two linear fits then give all five parameters:
$$\mu_g = c_0 + c_1\,\mathrm{VCD} + c_2\,\mathrm{IVCD} \;\Rightarrow\; \mu_{\max}=c_0,\ \mathrm{VCD}_{\max}=-\tfrac{c_0}{c_1},\ \mathrm{IVCD}_{\max}=-\tfrac{c_0}{c_2}$$
$$k_{d,\text{obs}} = k_{d,0} + k_{d,1}\,\mathrm{IVCD}$$

In [ ]:
pl.rate_panels(rates.dropna(subset=["mu_g"]), "mu_g", ["vcd", "ivcd"], r"$\mu_{g}$ [1/day]") # gross growth vs density and load
pl.rate_panels(rates.dropna(subset=["kd_obs"]), "kd_obs", ["ivcd", "elapsed_time"], r"$k_{d,\mathrm{obs}}$ [1/day]") # death vs load and time

In [ ]:
r2 = run("M2", lambda: mdl.CellModel(joint=False)) # benchmark + submission
pl.model_viewer(data, preds) # interactive viewer

**Result.** The shape is now right: growth, plateau, decline, and TCD separating from VCD. But daily rates are noisy, because they are differences of noisy measurements. Parameters fitted on rates do not give the best curves.

## M3: Cell model, joint fit
**What is new.** Same equations as M2. The five parameters are now fitted directly on the simulated curves of all training batches together, starting from the M2 values:
$$\min_{\theta}\ \sum_{b}\sum_{k}\left[\left(\frac{\widehat{\mathrm{VCD}}_b(t_k)-\mathrm{VCD}_b(t_k)}{\mathrm{VCD}_b(t_k)}\right)^2+\left(\frac{\widehat{\mathrm{TCD}}_b(t_k)-\mathrm{TCD}_b(t_k)}{\mathrm{TCD}_b(t_k)}\right)^2\right]$$
- $\theta = \ln(\mu_{\max}, \mathrm{VCD}_{\max}, \mathrm{IVCD}_{\max}, k_{d,0}, k_{d,1})$; logs keep the parameters positive;
- relative errors give the same weight to early, small values and late, large ones.

In [ ]:
r3 = run("M3", lambda: mdl.CellModel(joint=True), "submission_M3_cells_joint.csv") # benchmark + submission
display(pd.DataFrame({"M2 (daily rates)": r2["model"].p, "M3 (joint fit)": r3["model"].p})) # parameters compared
pl.model_viewer(data, preds) # interactive viewer

**Result.** M3 ≈ M0. With a single set of parameters, the model reproduces the *average* behaviour very well. But it has no way to know why one batch reaches 30 and another 15. Most of the error is a batch-to-batch offset: about half of the variance is a constant shift per batch. The equations cannot explain it from day 0 alone.

## M4: Cells + metabolite balances
**What is new.** Mass balances for substrate and byproduct, coupled to the cell model. Amounts are used ($A_S = S\,V$, $A_P = P\,V$), so dilution is automatic. Feeds containing substrate add a bolus.

**Equations** ($\mu^+ = \max(\mu, 0)$):
$$\frac{dA_S}{dt} = -q_S\,N, \qquad q_S = \left(\frac{\mu^+}{Y_{X/S}} + m_S\right)\frac{S}{K_S + S}$$
$$\frac{dA_P}{dt} = q_P\,N, \qquad q_P = \alpha\,\mu^+ - \beta\,P$$
At each feed time $t_f$: $A_S \leftarrow A_S + c_F\,F(t_f)$, with $c_F = c_{FS}$ for `feed_substrate` and $c_F = c_{FB}$ for `feed_b`.

| Parameter | Meaning |
|---|---|
| $Y_{X/S}$ | cells produced per gram of substrate |
| $m_S$ | maintenance consumption per cell |
| $K_S$ | substrate level at which uptake is half of its maximum |
| $c_{FS},\ c_{FB}$ | substrate concentration in the two feeds |
| $\alpha$ | byproduct produced per unit of growth |
| $\beta$ | byproduct re-consumption rate |

The cell parameters are taken from M3. The substrate parameters are fitted on substrate, then the byproduct parameters on byproduct. Takes 1–2 minutes.

In [ ]:
r4 = run("M4", lambda: mdl.CellMetaboliteModel(), "submission_M4_cells_metabolites.csv") # benchmark + submission
display(pd.Series(r4["model"].p).to_frame("M4 parameters")) # fitted parameters
pl.model_viewer(data, preds) # interactive viewer

**Result: a documented failure.** Substrate and byproduct are much worse than the mean curve. What the data show instead:
- **Overflow metabolism.** In days 0–4 the byproduct produced is about equal to the substrate consumed (≈ 1:1). The rate is per litre, not per cell, so a per-cell uptake law cannot fit it.
- **Unknown feed composition.** Feed B is dosed in proportion to the measured cells. Its substrate content, and the meaning of feeds C and D, are not given.

Without the feed composition and the real metabolic law, the balances add error instead of information. From now on, substrate and byproduct are handled by the data-driven correction.

## M5: Hybrid, mechanistic + ML
**Idea.** Keep M3 as the physical backbone. Learn its systematic errors from batch information that the equations do not use.

**Equations.** For target $c$ and day $d$:
$$\hat y_{b,c}(d) = \underbrace{y^{\text{M3}}_{b,c}(d)}_{\text{mechanistic}} + \underbrace{f_{c,d}(\mathbf{x}_b)}_{\text{learned correction}}$$
$f_{c,d}$ is a ridge regression trained on the training residuals $r_{b,c}(d) = y_{b,c}(d) - y^{\text{M3}}_{b,c}(d)$:
$$f_{c,d}(\mathbf{x}) = w_0 + \sum_j w_j\,z_j, \qquad z_j = \frac{x_j - \bar x_j}{s_j}$$
$$\hat{\mathbf w} = \arg\min_{\mathbf w}\ \sum_b \big(r_{b,c}(d) - f_{c,d}(\mathbf x_b)\big)^2 + \lambda\sum_j w_j^2$$
- one small model per target and day: $4 \times 13 = 52$ models;
- features are standardized ($z_j$: mean 0, std 1), so the coefficients are comparable;
- $\lambda$ (strength of the shrinkage) is chosen by internal cross-validation; with only 39 batches, shrinkage avoids overfitting.

**Features** $\mathbf x_b$, all known before the run (feeds are scaled by the volume):

| Feature | Meaning |
|---|---|
| `vcd0`, `substrate0`, `byproduct0`, `volume0` | day-0 sample |
| `fb_tot` | total feed B |
| `fc_n` | number of feed C additions |
| `fs_tot` | total substrate feed |
| `fs_t1` | time of the first substrate feed |
| `fs_24h`, `fb_24h` | substrate / feed B added in the 24 h before the sample (substrate only) |
| `fs_cum` | cumulative substrate fed until the sample (substrate only) |

In [ ]:
r5 = run("M5", lambda: mdl.RidgeCorrection(mdl.CellModel(joint=True), mdl.FEATS_BASIC, "M5 Hybrid: mechanistic + ML"), "submission_M5_hybrid.csv", show=("M3", "M5", "M6", "M7")) # benchmark + submission
pl.model_viewer(data, {k: v for k, v in preds.items() if k.startswith(("M3", "M5", "M6", "M7"))}) # only M3 and the hybrids

### What did the correction learn?
Each bar is one coefficient $w_j$ of the correction, averaged over the 13 days.

**How to read a bar.** All features are standardized, so $w_j$ is in the *units of the target*. It is the average change of the correction when feature $j$ increases by one standard deviation across batches, with the other features fixed.
- **Teal (positive):** batches with a high value of this feature end up *above* what M3 predicts.
- **Red (negative):** they end up *below*.

*Example:* the bar of about −0.25 on VCD for `fc_n` means: a batch with one standard deviation more feed C additions has, averaged over the 13 days, about $0.25\times10^6$ cells/mL *fewer* than M3 predicts.

**Why the bars look small.**
- The average over days includes the first days, where all batches are still close together and the corrections are near zero.
- Several correlated features share the same effect.

The bars show *which* features the correction uses and in which direction. They do not show the size of the effect on a single day.

**Caveats.**
- These are associations, not causes.
- Correlated features share their effect, and ridge shrinks all coefficients toward 0.
- The averages over days hide that some effects change sign over time.

In [ ]:
units = {"vcd": "10⁶ cells/mL", "tcd": "10⁶ cells/mL", "substrate": "g/L", "byproduct": "g/L"} # target units
pl.coefficient_bars(r5["model"].coefficients(), units) # mean standardized coefficients

**Result and interpretation.**
- The correction improves all four targets.
- Many feed C additions go with fewer cells and less byproduct. Hypothesis: feed C is the pH control (base added when acid or CO₂ accumulates), so it is a *symptom* of the culture state.
- Feed B is dosed on the measured cells, so its total carries information about how well the batch grew.

## M5b: Data-driven, mean + ML (control)
**Question.** Does the mechanistic backbone help, or would the same correction on the plain mean curve do equally well?

Same features and same ridge models as M5. Only the base changes:
$$\hat y_{b,c}(d) = \bar y_c(d) + f_{c,d}(\mathbf x_b)$$

In [ ]:
r5b = run("M5b", lambda: mdl.RidgeCorrection(mdl.MeanCurve(), mdl.FEATS_BASIC, "M5b Data-driven: mean + ML"), "submission_M5b_data_driven.csv") # benchmark + submission
pl.model_viewer(data, preds) # interactive viewer

**Result.** M5b ≈ M5. With a single global parameter set, M3 differs from batch to batch only through the day-0 values and the volume. Those are already among the features, so a linear correction can reproduce it.

**Lesson.** To add value, physics must carry *batch-specific* information. The correction alone cannot build it. This motivates M6.

## M6: Grey-box capacity + ML
**What is new.** The capacity $\mathrm{VCD}_{\max}$ becomes a property of each batch. It is predicted from the batch features, then fed back into the cell equations.

**Steps.**
1. Fit M3 (global parameters).
2. For each training batch, refit only $\mathrm{VCD}_{\max,b}$, with the other four parameters fixed. The fit is bounded within a factor $e$ of the global value.
3. Learn the capacity from the features (log scale keeps it positive):
$$\ln \mathrm{VCD}_{\max,b} = a_0 + \sum_j a_j\,z_{b,j} \qquad (\text{ridge, batch features})$$
4. For a new batch: predict $\mathrm{VCD}_{\max,b}$, simulate the cell equations, add the ridge correction of M5 on top.

The physics turns *"this batch will grow more"* into a full, consistent VCD and TCD trajectory: growth, peak, death, dilution.

In [ ]:
r6 = run("M6", lambda: mdl.RidgeCorrection(mdl.GreyBoxCapacity(), mdl.FEATS_BASIC, "M6 Grey-box capacity + ML"), "submission_M6_greybox.csv", show=("M3", "M5", "M6", "M7")) # benchmark + submission
gb = r6["model"].base # fitted grey-box part
fit_v = gb.vcd_max_fit # VCD_max fitted per training batch
pred_v = gb.vcd_max(data, list(fit_v.index)) # VCD_max predicted from features
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True) # two panels
axes[0].hist(fit_v, bins=15, color="#0B4F5C", alpha=0.85) # distribution
axes[0].axvline(gb.cells.p["vcd_max"], color="#E4572E", lw=2, label="global M3 value") # global value
axes[0].set(xlabel=r"fitted $\mathrm{VCD}_{\max}$ [10⁶ cells/mL]", ylabel="number of batches", title="capacity differs between batches") # labels
axes[0].legend() # legend
axes[1].scatter(fit_v, pred_v, color="#2A9D8F", s=25) # fitted vs predicted
lim = [fit_v.min() * 0.95, fit_v.max() * 1.05] # 1:1 range
axes[1].plot(lim, lim, color="black", lw=0.8) # 1:1 line
axes[1].set(xlabel=r"fitted $\mathrm{VCD}_{\max}$ [10⁶ cells/mL]", ylabel=r"predicted $\mathrm{VCD}_{\max}$ from features", title=f"r = {np.corrcoef(fit_v, pred_v)[0, 1]:.2f} (training batches)") # labels
plt.show() # draw
pl.model_viewer(data, {k: v for k, v in preds.items() if k.startswith(("M3", "M5", "M6", "M7"))}) # only M3 and the hybrids

**Result.** A small but consistent gain on VCD and TCD, the targets where the physics acts. The scatter shows that only part of the capacity differences can be predicted from what is known at day 0.

## M7: Grey-box + feed shape
**What is new.** Feed B is dosed on the measured cells, so the *shape* of the feed B profile partly reveals how the culture grew. Feed C timing marks when the culture starts to acidify. M7 adds these features, used only for substrate, where they help:

| Feature | Meaning |
|---|---|
| `fb_ramp` | growth rate of the feed B dose between day 1 and day 4: $\tfrac13\ln\!\big(\bar F_B(4)/\bar F_B(1)\big)$ |
| `fb_plateau` | mean feed B dose per volume, days 6–9 |
| `fb_late` | mean feed B dose per volume, days 10–12 |
| `fc_early` | number of feed C additions before day 5 |
| `fc_t1` | time of the first feed C addition |
| `viab0` | viability at day 0, $\mathrm{VCD}/\mathrm{TCD}$ |

Everything else as in M6.

In [ ]:
r7 = run("M7", lambda: mdl.RidgeCorrection(mdl.GreyBoxCapacity(), mdl.FEATS_SHAPE, "M7 Grey-box + feed shape"), "submission_M7_greybox_feedshape.csv", show=("M3", "M5", "M6", "M7")) # benchmark + submission
pl.model_viewer(data, {k: v for k, v in preds.items() if k.startswith(("M3", "M5", "M6", "M7"))}) # only M3 and the hybrids

## Summary
### Benchmark of all models

In [ ]:
table = ev.results_table(results) # all results
display(table) # table
pl.score_bars(table) # chart

### Robustness: 5-fold cross-validation
The benchmark uses one split of 10 batches, so small differences can be luck. In 5-fold cross-validation, every one of the 49 labeled batches is used as test once, and the five scores are averaged. M4 is skipped (slow, and clearly worse).

In [ ]:
cv_models = {"M0": lambda: mdl.MeanCurve(), # mean curve
             "M3": lambda: mdl.CellModel(joint=True), # cell model
             "M5": lambda: mdl.RidgeCorrection(mdl.CellModel(joint=True), mdl.FEATS_BASIC, "M5"), # hybrid
             "M5b": lambda: mdl.RidgeCorrection(mdl.MeanCurve(), mdl.FEATS_BASIC, "M5b"), # data-driven
             "M6": lambda: mdl.RidgeCorrection(mdl.GreyBoxCapacity(), mdl.FEATS_BASIC, "M6"), # grey-box
             "M7": lambda: mdl.RidgeCorrection(mdl.GreyBoxCapacity(), mdl.FEATS_SHAPE, "M7")} # grey-box + feed shape
cv = {k: ev.cross_validate(f, data) for k, f in cv_models.items()} # run CV (a few minutes)
cv_table = pd.DataFrame({k: {**pt.to_dict(), "score": s} for k, (s, pt) in cv.items()}).T # models x (targets + score)
display(pd.concat({"benchmark": table.loc[cv_table.index, "score"], "5-fold CV": cv_table["score"]}, axis=1)) # both scores

### Where the best model still fails
Error of M7 on each benchmark test batch and target. A few batches dominate the score. Their curves are far from the average, and nothing in the day-0 data announces it.

In [ ]:
pl.batch_heatmap(r7["per_batch"], "M7: error per test batch and target") # per-batch errors

### Conclusions
1. **Physics captures the average shape.**
   - Growth is limited by density and by the cumulative load, not by substrate.
   - The VCD decline after the peak is mostly dilution.
   - Alone, physics does not beat the mean curve (M3 ≈ M0).
2. **Metabolites do not follow simple mass balances (M4).** Overflow metabolism and the unknown feed compositions make them a data-driven problem.
3. **The learned correction helps (M5, ≈ −11%).** Whether the base is mechanistic or the mean curve makes no difference (M5b), unless the physics carries batch-specific information.
4. **Grey-box capacity (M6) and feed-shape features (M7) give the best and most robust score.** This holds on the benchmark and in cross-validation.
5. **The limit is information, not the algorithm.**
   - About half of the batch variance is a constant offset per batch that day 0 does not reveal.
   - More complex ML is unlikely to help without more information. Examples: the feed compositions, the meaning of feeds C and D, pH data.

**Recommended submission:** `outputs/submission_M7_greybox_feedshape.csv`.

All submission files (fitted on the 49 labeled batches, 156 rows each):

In [ ]:
for f in sorted(OUT.glob("submission_M*.csv")): # each submission file
    s = pd.read_csv(f) # load
    print(f"{f.name:45s} rows = {len(s)}, missing = {int(s.isna().sum().sum())}") # check